# 实验一：环境安装与 ECAPA-TDNN 初次推理（15%）

本实验的目标是让你第一次运行现代神经网络说话人识别模型。我们不会从零实现 ECAPA-TDNN，而是直接使用一个已经在中文说话人数据集 CN-Celeb 上训练好的 SpeechBrain checkpoint：`LanceaKing/spkrec-ecapa-cnceleb`。

通过本实验，你将完成以下任务：

- 成功配置并运行 SpeechBrain 环境；
- 读取一段示例语音 waveform；
- 使用预训练 ECAPA-TDNN 模型提取说话人嵌入，即 speaker embedding；
- 初步理解一段语音如何被神经网络转换为固定长度的说话人表征。

---

ECAPA-TDNN 论文：[Arxiv](https://arxiv.org/abs/2005.07143)

ECAPA-TDNN 是一种经典的现代说话人表征模型。它在 x-vector / TDNN 系列模型的基础上，引入了 1D Res2Net 模块、Squeeze-and-Excitation 机制、层级特征聚合，以及 channel-dependent frame attention，从而提升模型对说话人身份信息的建模能力。

## 安装uv
1. for Linux/macOS
```bash
curl -LsSf https://astral.sh/uv/install.sh | sh
```
2. for Windows(PowerShell)
```powershell
powershell -c "irm https://astral.sh/uv/install.ps1 | iex"
```

## 拉取 Speechbrain 项目代码
```bash
git clone https://github.com/speechbrain/speechbrain.git
```

## 安装项目依赖
1. 自动安装项目依赖
```bash
uv sync
```

> 所有code执行都必须使用 uv run xxx.py
>
> 因为有本地editable依赖，uv sync可能不成功。解决方法请参考2

2. 手动安装项目依赖 (uv)
```bash
rm -r .python-version pyproject.toml uv.lock .venv
uv init .
rm hello.py README.md
uv python pin 3.10
uv add --editable speechbrain/
uv add ipykernel pandas torch torchaudio torchcodec huggingface-hub matplotlib tqdm scikit-learn
```
> 所有code执行都必须使用 uv run xxx.py

3. 手动安装项目依赖 (conda, pip)
```bash
conda create -n speechbrain python=3.10 -y
conda activate speechbrain
pip install -e speechbrain/
pip install ipykernel pandas torch torchaudio torchcodec huggingface-hub matplotlib tqdm scikit-learn
```

In [1]:
# 运行环境检测
# 如果你无法使用 CUDA，仍然可以完成实验一到实验三的必做部分，只是推理速度会较慢
import sys
from pathlib import Path

import torch
import torchaudio
import speechbrain

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Torchaudio:", torchaudio.__version__)
print("Speechbrain:", getattr(speechbrain, "__version__", "unknown"))

cuda_available = torch.cuda.is_available()

if cuda_available:
    device = "cuda"
    print("GPU:", torch.cuda.get_device_name(0))
else:
    device = "cpu"
    print("GPU: CPU fallback")

print("Device:", device)

Python: 3.14.6 | packaged by conda-forge | (main, Jun 12 2026, 08:59:43) [MSC v.1944 64 bit (AMD64)]
Torch: 2.11.0+cu126
Torchaudio: 2.11.0+cu126
Speechbrain: unknown
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
Device: cuda


In [2]:
# 从 Huggingface 拉取 ECAPA-TDNN 模型
import os
# os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com' # 国内访问加速镜像

import huggingface_hub  
huggingface_hub.snapshot_download(repo_id="LanceaKing/spkrec-ecapa-cnceleb", local_dir="ckpt/spkrec-ecapa-cnceleb")

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

'D:\\Project\\SpeechInformationProcessing\\Homework\\Hw5\\ckpt\\spkrec-ecapa-cnceleb'

In [3]:
# 项目路径检查
# 养成检查路径的习惯，减少后续 FileNotFoundError。
from pathlib import Path

PROJECT_ROOT = Path(".").resolve()
CKPT_DIR = PROJECT_ROOT / "ckpt" / "spkrec-ecapa-cnceleb"
DATA_DIR = PROJECT_ROOT / "data" / "aishell_mini"
OUT_DIR = PROJECT_ROOT / "outputs"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CKPT_DIR exists:", CKPT_DIR.exists())
print("DATA_DIR exists:", DATA_DIR.exists())

for sub in ["embeddings", "scores", "figures", "reports"]:
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)

PROJECT_ROOT: D:\Project\SpeechInformationProcessing\Homework\Hw5
CKPT_DIR exists: True
DATA_DIR exists: True


In [5]:
# 加载 CN-Celeb ECAPA-TDNN 预训练模型
from pathlib import Path
import torchaudio
from speechbrain.inference.speaker import EncoderClassifier

classifier = EncoderClassifier.from_hparams(
    source="LanceaKing/spkrec-ecapa-cnceleb", 
    savedir="ckpt/spkrec-ecapa-cnceleb", 
    run_opts={"device": device})

print("Model loaded on:", device)
print("Modules:", classifier.mods.keys())

d:\Project\SpeechInformationProcessing\Homework\Hw5\speechbrain\utils\parameter_transfer.py:234: UserWarning: Requested Pretrainer collection using symlinks on Windows. This might not work; see `LocalStrategy` documentation. Consider unsetting `collect_in` in Pretrainer to avoid symlinking altogether.
  warnings.warn(
d:\Project\SpeechInformationProcessing\Homework\Hw5\speechbrain\utils\fetching.py:153: UserWarning: Using SYMLINK strategy on Windows for fetching potentially requires elevated privileges and is not recommended. See `LocalStrategy` documentation.
  warnings.warn(
Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.


Model loaded on: cuda
Modules: dict_keys(['compute_features', 'mean_var_norm', 'embedding_model', 'classifier'])


## checkpoint 中的超参数
请注意 `hyperparams.yaml` 中的几个字段：

- `compute_features`: Fbank 特征计算；
- `mean_var_norm`: 输入特征归一化；
- `embedding_model`: ECAPA-TDNN 主体；
- `classifier`: 训练阶段用于说话人分类的分类头；
- `lin_neurons`: embedding 维度；
- `out_n_neurons`: 训练时的说话人类别数。

> `hyperparams.yaml` 使用的是 HyperPyYAML 格式。HyperPyYAML 是 SpeechBrain 配套使用的配置解析库，可以在 YAML 中定义模块、对象和变量引用，从而把模型结构、路径和运行参数集中写在一个配置文件里。[Documentation](https://speechbrain.readthedocs.io/en/latest/API/hyperpyyaml.html)

In [6]:
# 查看 checkpoint 的 hyperparams
from pathlib import Path

hparam_path = CKPT_DIR / "hyperparams.yaml"
print(hparam_path)

with open(hparam_path, "r", encoding="utf-8") as f:
    text = f.read()

print(text[:2000])

D:\Project\SpeechInformationProcessing\Homework\Hw5\ckpt\spkrec-ecapa-cnceleb\hyperparams.yaml
# ############################################################################
# Model: ECAPA big for Speaker verification
# ############################################################################

# Feature parameters
n_mels: 80

# Pretrain folder (HuggingFace)
pretrained_path: LanceaKing/spkrec-ecapa-cnceleb

# Output parameters
out_n_neurons: 2793

# Model params
compute_features: !new:speechbrain.lobes.features.Fbank
    n_mels: !ref <n_mels>

mean_var_norm: !new:speechbrain.processing.features.InputNormalization
    norm_type: sentence
    std_norm: False

embedding_model: !new:speechbrain.lobes.models.ECAPA_TDNN.ECAPA_TDNN
    input_size: !ref <n_mels>
    channels: [1024, 1024, 1024, 1024, 3072]
    kernel_sizes: [5, 3, 3, 3, 1]
    dilations: [1, 2, 3, 4, 1]
    attention_channels: 128
    lin_neurons: 192

classifier: !new:speechbrain.lobes.models.ECAPA_TDNN.Classifier
    input

## 读取示例音频

In [7]:
# 读取示例音频

import torchaudio
import IPython.display as ipd

example_wav = "./data/sample.wav"

signal, sr = torchaudio.load(example_wav)

print("signal dtype:", signal.dtype)
print("signal shape:", signal.shape)
print("sample rate:", sr)
print("min:", signal.min().item())
print("max:", signal.max().item())
print("duration sec:", signal.shape[-1] / sr)

ipd.Audio(signal[0].numpy(), rate=sr)

signal dtype: torch.float32
signal shape: torch.Size([1, 95984])
sample rate: 16000
min: -0.041259765625
max: 0.047088623046875
duration sec: 5.999


## 用 `encode_batch()` 提取 embedding
SpeechBrain 的 `encode_batch()` 源码流程是：必要时给一维 waveform 增加 batch 维度；如果未传 `wav_lens`，使用全 1 长度；把 waveform 移到 `device` 上并转成 float；依次执行 `compute_features`、`mean_var_norm`、`embedding_model`，最后可选执行 embedding 级别归一化。

In [8]:
# 推理得到embedding
import torch
import torch.nn.functional as F

@torch.no_grad()
def extract_embedding(signal, sr):
    # torchaudio.load gives [channel, time]; SpeechBrain expects [batch, time]
    wavs = signal.squeeze(0).unsqueeze(0).to(device)

    emb = classifier.encode_batch(wavs)
    emb = emb.squeeze().detach().cpu()

    emb_l2 = F.normalize(emb, dim=0)

    return emb, emb_l2

emb, emb_l2 = extract_embedding(signal, sr)

print("embedding shape:", emb.shape)
print("raw embedding norm:", emb.norm().item())
print("L2-normalized embedding norm:", emb_l2.norm().item())

embedding shape: torch.Size([192])
raw embedding norm: 234.6773223876953
L2-normalized embedding norm: 1.0


In [9]:
# 保存 embedding
import numpy as np

out_path = OUT_DIR / "embeddings" / "example1_embedding.npy"
np.save(out_path, emb_l2.numpy())

print("Saved:", out_path)


Saved: D:\Project\SpeechInformationProcessing\Homework\Hw5\outputs\embeddings\example1_embedding.npy


In [10]:
# 对比 SciPy 读取结果，送入模型进行推理会如何？
from scipy.io import wavfile

rate, data = wavfile.read(example_wav)

print("scipy sample rate:", rate)
print("scipy dtype:", data.dtype)
print("scipy shape:", data.shape)
print("scipy min/max:", data.min(), data.max())

# ... YOUR CODE HERE

scipy sample rate: 16000
scipy dtype: int16
scipy shape: (95984,)
scipy min/max: -1352 1543


In [15]:
@torch.no_grad()
def extract_embedding_test(signal, sr):
    # torchaudio.load gives [channel, time]; SpeechBrain expects [batch, time]
    wavs = torch.from_numpy(signal).to(device)

    emb = classifier.encode_batch(wavs)
    emb = emb.squeeze().detach().cpu()

    emb_l2 = F.normalize(emb, dim=0)

    return emb, emb_l2

emb, emb_l2 = extract_embedding_test(data, rate)

print("embedding shape:", emb.shape)
print("raw embedding norm:", emb.norm().item())
print("L2-normalized embedding norm:", emb_l2.norm().item())

embedding shape: torch.Size([192])
raw embedding norm: 234.6703338623047
L2-normalized embedding norm: 1.0


## 思考题
Q1. waveform 读取与数值范围

查阅 [Torchaudio Doc](https://docs.pytorch.org/audio/stable/index.html)，结合本实验输出，回答：
- torchaudio.load() 默认返回的 signal dtype 是什么？
- 默认返回的 signal shape 是什么？[channel, time] 还是 [time, channel]？
- 使用 SciPy 包的 `scipy.io.wavfile.read()` 方法也可以读取音频。对于 16-bit PCM WAV，torchaudio.load() 和 scipy.io.wavfile.read() 得到的数据 dtype 和数值范围分别是什么？
- 如果把 scipy.io.wavfile.read() 得到的 int16 数据不缩放，直接转成 Tensor 输入 encode_batch()，你预期会出现什么问题？请用一小段实验验证你的判断。

> Hint
> 直接输入 int16 不一定会报错，因为 encode_batch() 内部会把 waveform 转成 float。但“不报错”不等于“输入正确”。模型期望的是合理尺度的 waveform。

Q2. SpeechBrain checkpoint 结构

查阅 [Speechbrain Doc](https://speechbrain.readthedocs.io/en/latest) 和本地 ckpt/spkrec-ecapa-cnceleb/hyperparams.yaml，回答：

- 本 checkpoint 中 modules 包含哪些模块？
- compute_features、mean_var_norm、embedding_model、classifier 分别有什么用途？
- lin_neurons=192 表示什么？
- out_n_neurons=2793 表示什么？
- encode_batch() 是否会使用 classifier？如果不会，classifier 在训练和 classify_batch() 中的作用是什么？[hint](speechbrain/recipes/VoxCeleb/SpeakerRec/train_speaker_embeddings.py)

Q3. embedding 每个维度是否有明确语义？

`embeddings` 的 shape 是 [1, 1, 192] 或 [192]。请解释：
- 这 192 个 dimension 是否分别代表“性别”“音高”“口音”“年龄”等人工可解释属性？
- 如果不是，为什么我们仍然可以用这个向量做说话人比对？
- 如果某些维度和性别、口音高度相关，这对系统公平性和隐私有什么潜在影响？